# Macrocosm photo-z CNN - train your own version (Colab + MLflow)

**One self-contained notebook.** It loads the (cropped) training set **into RAM once** and trains
from memory - fast, no per-epoch disk I/O. At **32px the full 600k fits standard Colab (~6 GB)**;
for 64px use a subset or a High-RAM runtime (see section 1).

**To make your own version:** edit the **MODEL cell (section 2)**, name your run in the MLflow cell,
run all. **Runtime -> Change runtime type -> GPU** first.

## 0. Setup (Colab) - clone repo, install, log in, download all data
Clones **CNN-Model** (brings `eval.py` + `splits/`), installs deps, downloads all image shards.

In [ ]:
![ -d /content/CNN-Model ] || git clone https://github.com/Le-Wagon-Macrocosm/CNN-Model.git /content/CNN-Model
%cd /content/CNN-Model
!pip -q install mlflow
from google.colab import auth; auth.authenticate_user()
!gcloud config set project macrocosm-lewagon -q
!mkdir -p /content/data
!gcloud storage cp gs://macrocosm-lewagon/data/sample_v1/catalog_v1.parquet /content/data/

DATA_DIR = '/content/data'

In [ ]:
!gcloud storage cp gs://macrocosm-lewagon/data/sample_v1/images_*.npy /content/data/

## 1. Load the training data into RAM
Train only on objids from `splits/train_objids.csv` (never the 50k val - that keeps `eval.py`
honest); `is_train_subset` checks it. **`CROP`** sets the cutout size loaded into RAM:
- `CROP=32` -> full 600k fits standard Colab (~6 GB). **Default.**
- `CROP=64` -> set `N=150000` (a subset, ~6 GB) OR use a **High-RAM runtime** for the full 600k (~22 GB).

In [ ]:
import glob, re, numpy as np, pandas as pd
from eval import is_train_subset
SHARD = 6000
DATA_DIR = globals().get('DATA_DIR', '/content/data')
CROP = 64#globals().get('CROP', 32)                      # 32 (full 600k) or 64 (subset / High-RAM)

TRAIN_CSV = globals().get('TRAIN_CSV', 'splits/train_objids.csv')   # or your own subset csv
chk = is_train_subset(TRAIN_CSV)
assert chk['ok'], f"LEAK: {chk['n_outside_train']} objids in {TRAIN_CSV} are NOT in the train split {chk['sample_outside']}"
print(f"train csv OK: {chk['n']:,} objids")

cat = pd.read_parquet(f'{DATA_DIR}/catalog_v1.parquet', columns=['objid', 'redshift'])
z_all = cat['redshift'].values
o2i = {int(o): i for i, o in enumerate(cat['objid'].values)}
idx = np.array([o2i[int(o)] for o in pd.read_csv(TRAIN_CSV)['objid'].values], dtype=np.int64)
present = set(int(re.findall(r'images_(\d+)_', p)[0]) // SHARD for p in glob.glob(f'{DATA_DIR}/images_*.npy'))
idx = idx[[(i // SHARD) in present for i in idx]]
N = globals().get('N', len(idx))                      # cap for 64px or a quick run
rng = np.random.RandomState(0); rng.shuffle(idx); idx = idx[:N]
es_idx, train_idx = idx[:5000], idx[5000:]            # early-stop set held out FROM train

def load_into_ram(rows, crop):
    """Center-crop + load these rows into a float16 array in RAM (sequential reads per shard)."""
    mm = {int(re.findall(r'images_(\d+)_', p)[0]) // SHARD: np.load(p, mmap_mode='r')
          for p in glob.glob(f'{DATA_DIR}/images_*.npy')}
    o = (64 - crop) // 2; rows = np.asarray(rows, np.int64)
    X = np.empty((len(rows), crop, crop, 5), np.float16)
    for s in np.unique(rows // SHARD):
        sel = np.where(rows // SHARD == s)[0]; rr = rows[sel] % SHARD; srt = np.argsort(rr)
        X[sel[srt]] = mm[int(s)][rr[srt]][:, o:o+crop, o:o+crop, :]   # sorted -> sequential read
    return X, np.log1p(z_all[rows]).astype('float32')

print('loading train into RAM (one-time)...')
Xtr, ytr = load_into_ram(train_idx, CROP)
Xes, _   = load_into_ram(es_idx, CROP)
zes = z_all[es_idx]
print(f'train {Xtr.shape} ({Xtr.nbytes/1e9:.1f} GB float16) / early-stop {Xes.shape}')

## 2. THE MODEL - **edit this cell** to redesign the architecture
Example: VGG stem + 3 lightweight **Inception** modules -> GAP -> a 64-d `embedding` (feeds the
fusion head later) -> `z`. GlobalAveragePooling makes it size-agnostic, so the same code works at
32px or 64px. Change anything: depth, Inception blocks, a classification head, transfer learning.
(KB MCM-A-18). The input shape follows `CROP`.

In [ ]:
from tensorflow.keras import layers as L, Model, Input
from tensorflow.keras import regularizers

def inception(x, f1, f3r, f3, f5r, f5, fp, name, reg=None):
    b1 = L.Conv2D(f1, 1, padding='same', activation='relu', kernel_regularizer=reg, name=f'{name}_1x1')(x)
    b3 = L.Conv2D(f3r, 1, padding='same', activation='relu', kernel_regularizer=reg, name=f'{name}_3x3_reduce')(x)
    b3 = L.Conv2D(f3, 3, padding='same', activation='relu', kernel_regularizer=reg, name=f'{name}_3x3')(b3)
    b5 = L.Conv2D(f5r, 1, padding='same', activation='relu', kernel_regularizer=reg, name=f'{name}_5x5_reduce')(x)
    b5 = L.Conv2D(f5, 5, padding='same', activation='relu', kernel_regularizer=reg, name=f'{name}_5x5')(b5)
    bp = L.MaxPool2D(3, strides=1, padding='same', name=f'{name}_pool')(x)
    bp = L.Conv2D(fp, 1, padding='same', activation='relu', kernel_regularizer=reg, name=f'{name}_pool_proj')(bp)
    return L.Concatenate(axis=-1, name=f'{name}_concat')([b1, b3, b5, bp])

def build_cnn(input_shape, embed_dim=64, l2=1e-4, drop=0.4, spatial_drop=0.1):
    reg = regularizers.l2(l2) if l2 else None
    inp = Input(shape=input_shape, name='cutout')
    x = L.Conv2D(32, 3, padding='same', activation='relu', kernel_regularizer=reg, name='stem1a')(inp)
    x = L.Conv2D(32, 3, padding='same', activation='relu', kernel_regularizer=reg, name='stem1b')(x)
    x = L.BatchNormalization(name='stem1_bn')(x); x = L.MaxPool2D(name='stem1_pool')(x)
    x = L.Conv2D(64, 3, padding='same', activation='relu', kernel_regularizer=reg, name='stem2')(x)
    x = L.BatchNormalization(name='stem2_bn')(x); x = L.MaxPool2D(name='stem2_pool')(x)

    x = inception(x, 32, 32, 48, 8, 24, 24, name='inc1', reg=reg); x = L.BatchNormalization(name='inc1_bn')(x)
    x = L.SpatialDropout2D(spatial_drop, name='inc1_sdrop')(x)
    x = L.MaxPool2D(name='inc1_down')(x)
    x = inception(x, 64, 48, 96, 16, 48, 48, name='inc2', reg=reg); x = L.BatchNormalization(name='inc2_bn')(x)
    x = L.SpatialDropout2D(spatial_drop, name='inc2_sdrop')(x)
    x = inception(x, 64, 48, 96, 16, 48, 48, name='inc3', reg=reg); x = L.BatchNormalization(name='inc3_bn')(x)

    x = L.GlobalAveragePooling2D(name='gap')(x)
    x = L.Dense(128, activation='relu', kernel_regularizer=reg, name='dense')(x)
    x = L.Dropout(drop, name='dropout')(x)
    emb = L.Dense(embed_dim, activation='relu', kernel_regularizer=reg, name='embedding')(x)
    x = L.Dropout(drop, name='embedding_drop')(emb)          # extra dropout before the head
    zout = L.Dense(1, name='z')(x)
    return Model(inp, zout, name='photoz_cnn')

def build_embedder(cnn):
    return Model(cnn.input, cnn.get_layer('embedding').output, name='photoz_embedder')

model = build_cnn((CROP, CROP, 5))
model.summary()
print('params:', f'{model.count_params():,}')


## 3. Pipeline + metrics - usually leave as-is
In-RAM `tf.data` (from_tensor_slices): batch -> arcsinh + per-image norm -> augment. Shuffle buffer is bounded so RAM stays flat. sigma_MAD is the metric to beat.

In [ ]:
import tensorflow as tf

def preprocess(x, y):                      # x: (B,H,W,5) -> arcsinh + per-image per-channel norm
    x = tf.cast(x, tf.float32); x = tf.math.asinh(x)
    m = tf.reduce_mean(x, axis=[1, 2], keepdims=True)
    s = tf.math.reduce_std(x, axis=[1, 2], keepdims=True) + 1e-6
    return (x - m) / s, y

def augment(x, y):
    x = tf.image.rot90(x, tf.random.uniform([], 0, 4, dtype=tf.int32))
    x = tf.image.random_flip_left_right(x); x = tf.image.random_flip_up_down(x)
    return x, y

def ram_dataset(X, y, training=False, batch=256, shuffle_buf=50000):
    n = len(X); H, W = X.shape[1], X.shape[2]
    ds = tf.data.Dataset.range(n)
    if training: ds = ds.shuffle(min(n, shuffle_buf), reshuffle_each_iteration=True)
    ds = ds.batch(batch)
    def gather(i):                       # cast inside -> returned dtype always matches Tout
        xb = tf.numpy_function(lambda ii: X[ii].astype('float16'), [i], tf.float16)
        yb = tf.numpy_function(lambda ii: y[ii].astype('float32'), [i], tf.float32)
        xb.set_shape([None, H, W, 5]); yb.set_shape([None])
        return xb, yb
    ds = ds.map(gather, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if training: ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

class SigmaMadCallback(tf.keras.callbacks.Callback):
    def __init__(self, val_ds, z_true): super().__init__(); self.val_ds = val_ds; self.z_true = np.asarray(z_true)
    def on_epoch_end(self, epoch, logs=None):
        zp = np.expm1(self.model.predict(self.val_ds, verbose=0).ravel())
        sm, out = sigma_mad(self.z_true, zp), outlier_rate(self.z_true, zp)
        logs = logs if logs is not None else {}
        logs['val_sigma_MAD'] = sm; logs['val_outlier'] = out
        try:                       # log metrics OURSELVES (autolog misses them on Keras 3)
            import mlflow
            if mlflow.active_run():
                mlflow.log_metrics({k: float(v) for k, v in logs.items()}, step=epoch)
        except Exception as e:
            print('  (mlflow metric log skipped:', e, ')')
        print(f'  -> val sigma_MAD={sm:.4f}  outlier={out*100:.1f}%')

## 4. Connect MLflow (optional)
Paste the bearer token (ask the team); start the server first (`make mlflow-start`). Without a token it just trains and skips logging.

In [ ]:
import os
MLFLOW_TOKEN = 'PASTE_MLFLOW_API_TOKEN_HERE'
USE_MLFLOW = 'PASTE' not in MLFLOW_TOKEN
if USE_MLFLOW:
    import mlflow, mlflow.tensorflow
    os.environ['MLFLOW_TRACKING_URI'] = 'https://146-148-10-86.sslip.io'
    os.environ['MLFLOW_TRACKING_TOKEN'] = MLFLOW_TOKEN
    mlflow.set_experiment('photoz-cnn')
    mlflow.tensorflow.autolog(log_models=True, log_datasets=False)   # log_datasets=False avoids a hang
    print('MLflow: logging to', os.environ['MLFLOW_TRACKING_URI'])
else:
    print('MLflow token not set -> training without logging (fine for a quick test)')

## 5. Train + log
Trains from RAM (fast), early-stops on the held-out-from-train set, then scores the **canonical
50k val** via `eval.py` (`val50k_sigma_MAD` - the number we compare). Name **your** run. Logs the
`CONFIG` + a `recipe.py` artifact (exact preprocessing/model source) so the run is reproducible.

In [ ]:
import inspect
from eval import evaluate
EPOCHS = globals().get('EPOCHS', 50)
RUN_NAME = 'default_baseline-x3'
BATCH, LR = 256, 3e-4

train_ds = ram_dataset(Xtr, ytr, training=True, batch=BATCH)
es_ds    = ram_dataset(Xes, np.log1p(zes), training=False, batch=512)
model.compile(optimizer=tf.keras.optimizers.Adam(LR),
              loss=tf.keras.losses.Huber(delta=0.02), metrics=['mae'])
cbs = [SigmaMadCallback(es_ds, zes),
       tf.keras.callbacks.EarlyStopping('val_sigma_MAD', mode='min', patience=8, restore_best_weights=True),
       tf.keras.callbacks.ReduceLROnPlateau('val_sigma_MAD', mode='min', factor=0.5, patience=3, min_lr=1e-5)]
CONFIG = dict(crop=CROP, batch=BATCH, lr=LR, optimizer='adam', loss='huber(0.02)', target='log1p(z)',
              stretch='arcsinh', norm='per-image per-channel', augment='rot90+flip',
              arch='vgg-stem+3inception', train_csv=TRAIN_CSV, N=len(train_idx),
              params=int(model.count_params()))

def run_training():
    model.fit(train_ds, validation_data=es_ds, epochs=EPOCHS, callbacks=cbs)
    m = evaluate(model, data_dir=DATA_DIR, crop=CROP)   # CANONICAL score on the fixed 50k val
    print('50k val:', m)
    return m

if USE_MLFLOW:
    import mlflow
    with mlflow.start_run(run_name=RUN_NAME):
        mlflow.log_params(CONFIG)
        mlflow.log_text('\n\n'.join(inspect.getsource(f)
                        for f in (preprocess, augment, ram_dataset, build_cnn)), 'recipe.py')
        m = run_training()
        mlflow.log_metrics({'val50k_sigma_MAD': m['sigma_MAD'], 'val50k_outlier': m['outlier'], 'val50k_n': m['n']})
else:
    run_training()

## 6. Compare
Open the MLflow UI (`https://146-148-10-86.sslip.io`, GitHub-org login) and sort runs by
**`val50k_sigma_MAD`** - everyone scores on the same fixed 50k val, so it's directly comparable.
The lowest wins (its `embedding` feeds the fusion head).